# 第2回 発展課題：用語集の学習と検索の比較

学習前・学習後・用語検索を同じ42問で比較した。学習時は用語集の定義を教師回答とし、推論時にはシステム指示と質問だけを入力する。

用語集に未登録の語は、実在する語でも確認要求を正解とする。評価対象12語の定義は学習に含む。

元ノートブック：`notebooks/lora_format_tuning.ipynb`。


## 0. 実行環境

Google ColabのTesla T4を使用。モデルのrevisionと主要ライブラリの版を固定した。


In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.14.1+cu130'}


## 1. 学習設定

3エポック、学習率0.0002、LoRAランク16、生成上限256トークン。学習前後で同じ生成条件を使い、モデル・revision・主要ライブラリの固定版は元ノートブックを引き継いだ。


In [4]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 3,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/lora_glossary",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/lora_glossary",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

Mounted at /content/drive
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 3,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/lora_glossary",
  "drive_dir": "/content/drive/MyDrive/aie2026/lora_glossary"
}


## 2. 学習・検証・評価データ

講座用語集58語の定義から学習124件・検証62件を作成した。評価42問は通常12問・言い換え12問・別名12問・未登録語6問で構成する。未登録語は用語集の見出し・別名・IDにない語を基準にした。

質問文はtrain / valid / testで重複しないが、評価対象の用語と定義は学習に含む。testの正解は採点だけに使い、モデルの入力には渡さない。


In [5]:
PAYLOAD = json.loads(r"""{
  "task": "glossary",
  "system": "あなたは講座用語集の回答係です。対象は学習用の講座用語集に登録された用語だけです。必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。用語集にある語には定義を答え、needs_review を false にします。用語集にない語や定義がわからない場合は、answer に「確認が必要」と書き、needs_review を true にします。",
  "faq": {},
  "glossary": {
    "version": "2026-draft-1",
    "note": "AIE2026 第1〜3回・第5回の用語集（たたき台）。講義スライド確定後に内容を差し替える。",
    "terms": [
      {
        "id": "ai-engineering",
        "term": "AIエンジニアリング",
        "aliases": [
          "AI Engineering"
        ],
        "lecture": 1,
        "category": "全体像",
        "definition": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。",
        "related": [
          "llm",
          "evaluation-loop"
        ]
      },
      {
        "id": "evaluation-loop",
        "term": "評価と改善のサイクル",
        "aliases": [
          "改善サイクル"
        ],
        "lecture": 1,
        "category": "全体像",
        "definition": "作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。",
        "related": [
          "ai-engineering",
          "hallucination"
        ]
      },
      {
        "id": "llm",
        "term": "LLM（大規模言語モデル）",
        "aliases": [
          "大規模言語モデル",
          "Large Language Model"
        ],
        "lecture": 1,
        "category": "LLM基礎",
        "definition": "大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。",
        "related": [
          "token",
          "prompt",
          "pretraining"
        ]
      },
      {
        "id": "token",
        "term": "トークン",
        "aliases": [
          "token"
        ],
        "lecture": 1,
        "category": "LLM基礎",
        "definition": "LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。",
        "related": [
          "context-window",
          "rate-limit"
        ]
      },
      {
        "id": "context-window",
        "term": "コンテキストウィンドウ",
        "aliases": [
          "コンテキスト長",
          "context window"
        ],
        "lecture": 1,
        "category": "LLM基礎",
        "definition": "LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。",
        "related": [
          "token",
          "agent-loop"
        ]
      },
      {
        "id": "prompt",
        "term": "プロンプト",
        "aliases": [
          "prompt"
        ],
        "lecture": 1,
        "category": "LLM基礎",
        "definition": "LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。",
        "related": [
          "system-prompt",
          "llm"
        ]
      },
      {
        "id": "system-prompt",
        "term": "システムプロンプト",
        "aliases": [
          "指示文",
          "instructions",
          "system instruction"
        ],
        "lecture": 1,
        "category": "LLM基礎",
        "definition": "利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。",
        "related": [
          "prompt",
          "guardrail"
        ]
      },
      {
        "id": "temperature",
        "term": "temperature",
        "aliases": [
          "温度パラメータ"
        ],
        "lecture": 1,
        "category": "LLM基礎",
        "definition": "次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。",
        "related": [
          "llm",
          "evaluation-loop"
        ]
      },
      {
        "id": "hallucination",
        "term": "ハルシネーション",
        "aliases": [
          "幻覚",
          "hallucination"
        ],
        "lecture": 1,
        "category": "LLM基礎",
        "definition": "LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。",
        "related": [
          "tool",
          "evaluation-loop"
        ]
      },
      {
        "id": "api",
        "term": "API",
        "aliases": [
          "Application Programming Interface",
          "Web API"
        ],
        "lecture": 1,
        "category": "開発基礎",
        "definition": "プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。",
        "related": [
          "api-key",
          "rate-limit"
        ]
      },
      {
        "id": "api-key",
        "term": "APIキー",
        "aliases": [
          "API key"
        ],
        "lecture": 1,
        "category": "開発基礎",
        "definition": "APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。",
        "related": [
          "codespaces-secret",
          "environment-variable"
        ]
      },
      {
        "id": "environment-variable",
        "term": "環境変数",
        "aliases": [
          "environment variable",
          "env"
        ],
        "lecture": 1,
        "category": "開発基礎",
        "definition": "プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。",
        "related": [
          "api-key",
          "codespaces-secret"
        ]
      },
      {
        "id": "rate-limit",
        "term": "レート制限",
        "aliases": [
          "rate limit",
          "429エラー"
        ],
        "lecture": 1,
        "category": "開発基礎",
        "definition": "一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。",
        "related": [
          "api",
          "agent-loop",
          "default-model"
        ]
      },
      {
        "id": "codespaces",
        "term": "GitHub Codespaces",
        "aliases": [
          "Codespaces",
          "コードスペース"
        ],
        "lecture": 1,
        "category": "開発環境",
        "definition": "GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。",
        "related": [
          "devcontainer",
          "codespaces-secret"
        ]
      },
      {
        "id": "devcontainer",
        "term": "devcontainer",
        "aliases": [
          "Dev Container",
          "devcontainer.json"
        ],
        "lecture": 1,
        "category": "開発環境",
        "definition": "開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。",
        "related": [
          "codespaces"
        ]
      },
      {
        "id": "codespaces-secret",
        "term": "Codespaces secret",
        "aliases": [
          "Codespacesのシークレット"
        ],
        "lecture": 1,
        "category": "開発環境",
        "definition": "GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。",
        "related": [
          "api-key",
          "environment-variable",
          "codespaces"
        ]
      },
      {
        "id": "fork",
        "term": "fork",
        "aliases": [
          "フォーク"
        ],
        "lecture": 1,
        "category": "Git/GitHub",
        "definition": "他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。",
        "related": [
          "branch",
          "pull-request"
        ]
      },
      {
        "id": "branch",
        "term": "ブランチ",
        "aliases": [
          "branch"
        ],
        "lecture": 1,
        "category": "Git/GitHub",
        "definition": "作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。",
        "related": [
          "pull-request",
          "fork"
        ]
      },
      {
        "id": "pull-request",
        "term": "プルリクエスト（PR）",
        "aliases": [
          "PR",
          "Pull Request"
        ],
        "lecture": 1,
        "category": "Git/GitHub",
        "definition": "あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。",
        "related": [
          "branch",
          "answer-md"
        ]
      },
      {
        "id": "github-copilot",
        "term": "GitHub Copilot",
        "aliases": [
          "Copilot"
        ],
        "lecture": 1,
        "category": "AIコーディング",
        "definition": "GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。",
        "related": [
          "code-completion",
          "coding-agent"
        ]
      },
      {
        "id": "code-completion",
        "term": "コード補完",
        "aliases": [
          "インライン補完",
          "inline suggestion"
        ],
        "lecture": 1,
        "category": "AIコーディング",
        "definition": "書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。",
        "related": [
          "github-copilot"
        ]
      },
      {
        "id": "coding-agent",
        "term": "コーディングエージェント",
        "aliases": [
          "Coding Agent",
          "AIコーディングエージェント"
        ],
        "lecture": 1,
        "category": "AIコーディング",
        "definition": "ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。",
        "related": [
          "ai-agent",
          "harness"
        ]
      },
      {
        "id": "pretraining",
        "term": "事前学習",
        "aliases": [
          "pre-training",
          "プレトレーニング"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。",
        "related": [
          "fine-tuning",
          "llm"
        ]
      },
      {
        "id": "fine-tuning",
        "term": "ファインチューニング",
        "aliases": [
          "fine-tuning",
          "追加学習",
          "FT"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。",
        "related": [
          "sft",
          "peft",
          "lora"
        ]
      },
      {
        "id": "sft",
        "term": "教師ありファインチューニング（SFT）",
        "aliases": [
          "SFT",
          "Supervised Fine-Tuning",
          "指示チューニング"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。",
        "related": [
          "fine-tuning",
          "lora"
        ]
      },
      {
        "id": "peft",
        "term": "PEFT",
        "aliases": [
          "Parameter-Efficient Fine-Tuning",
          "パラメータ効率の良いファインチューニング"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。",
        "related": [
          "lora",
          "fine-tuning"
        ]
      },
      {
        "id": "lora",
        "term": "LoRA",
        "aliases": [
          "Low-Rank Adaptation"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。",
        "related": [
          "peft",
          "lora-rank",
          "qlora"
        ]
      },
      {
        "id": "lora-rank",
        "term": "ランク（LoRAのr）",
        "aliases": [
          "rank",
          "r"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。",
        "related": [
          "lora"
        ]
      },
      {
        "id": "qlora",
        "term": "QLoRA",
        "aliases": [
          "Quantized LoRA"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。",
        "related": [
          "lora",
          "quantization"
        ]
      },
      {
        "id": "quantization",
        "term": "量子化",
        "aliases": [
          "quantization"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。",
        "related": [
          "qlora"
        ]
      },
      {
        "id": "learning-rate",
        "term": "学習率",
        "aliases": [
          "learning rate",
          "lr"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。",
        "related": [
          "epoch",
          "overfitting"
        ]
      },
      {
        "id": "epoch",
        "term": "エポック",
        "aliases": [
          "epoch"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。",
        "related": [
          "overfitting",
          "learning-rate"
        ]
      },
      {
        "id": "overfitting",
        "term": "過学習",
        "aliases": [
          "overfitting",
          "オーバーフィッティング"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。",
        "related": [
          "epoch",
          "evaluation-loop"
        ]
      },
      {
        "id": "huggingface-hub",
        "term": "Hugging Face Hub",
        "aliases": [
          "Hugging Face",
          "HF Hub"
        ],
        "lecture": 2,
        "category": "ファインチューニング",
        "definition": "公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。",
        "related": [
          "lora",
          "pretraining"
        ]
      },
      {
        "id": "ai-agent",
        "term": "AIエージェント",
        "aliases": [
          "エージェント",
          "agent"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。",
        "related": [
          "agent-loop",
          "tool",
          "harness"
        ]
      },
      {
        "id": "agent-loop",
        "term": "エージェントループ",
        "aliases": [
          "agent loop",
          "ツール呼び出しループ"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。",
        "related": [
          "ai-agent",
          "tool",
          "rate-limit"
        ]
      },
      {
        "id": "tool",
        "term": "ツール（Tool use / Function calling）",
        "aliases": [
          "ツール呼び出し",
          "Function calling",
          "Tool use",
          "ツール"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。",
        "related": [
          "tool-description",
          "agent-loop"
        ]
      },
      {
        "id": "tool-description",
        "term": "ツールの説明文",
        "aliases": [
          "docstring",
          "ツールスキーマ",
          "tool schema"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。",
        "related": [
          "tool",
          "pydantic-ai"
        ]
      },
      {
        "id": "structured-output",
        "term": "構造化出力",
        "aliases": [
          "structured output",
          "JSON出力"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。",
        "related": [
          "pydantic-ai"
        ]
      },
      {
        "id": "harness",
        "term": "ハーネス",
        "aliases": [
          "agent harness",
          "エージェントハーネス"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。",
        "related": [
          "coding-agent",
          "pi"
        ]
      },
      {
        "id": "pi",
        "term": "Pi",
        "aliases": [
          "Pi coding agent",
          "pi agent"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。",
        "related": [
          "harness",
          "coding-agent"
        ]
      },
      {
        "id": "pydantic-ai",
        "term": "Pydantic AI",
        "aliases": [
          "pydantic-ai",
          "PydanticAI"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。",
        "related": [
          "test-model",
          "structured-output",
          "adk"
        ]
      },
      {
        "id": "adk",
        "term": "ADK（Agent Development Kit）",
        "aliases": [
          "ADK",
          "Agent Development Kit",
          "google-adk"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深い。2.0で1.x系から大きな変更が入っている。本講座では第3回の宿題の発展レベルで扱う。",
        "related": [
          "pydantic-ai",
          "ai-agent"
        ]
      },
      {
        "id": "mcp",
        "term": "MCP（Model Context Protocol）",
        "aliases": [
          "MCP",
          "Model Context Protocol"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。",
        "related": [
          "tool",
          "ai-agent",
          "mcp-server",
          "mcp-client"
        ]
      },
      {
        "id": "guardrail",
        "term": "ガードレール",
        "aliases": [
          "guardrail"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。",
        "related": [
          "human-in-the-loop",
          "system-prompt"
        ]
      },
      {
        "id": "human-in-the-loop",
        "term": "Human-in-the-loop",
        "aliases": [
          "HITL",
          "人間による承認",
          "確認を挟む"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。",
        "related": [
          "guardrail",
          "tool"
        ]
      },
      {
        "id": "test-model",
        "term": "TestModel",
        "aliases": [
          "テスト用モデル"
        ],
        "lecture": 3,
        "category": "エージェント",
        "definition": "Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。",
        "related": [
          "pydantic-ai",
          "rate-limit"
        ]
      },
      {
        "id": "homework-levels",
        "term": "宿題の3レベル",
        "aliases": [
          "3レベル",
          "宿題レベル"
        ],
        "lecture": 1,
        "category": "講座",
        "definition": "AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。",
        "related": [
          "answer-md"
        ]
      },
      {
        "id": "answer-md",
        "term": "answer.md",
        "aliases": [
          "解答ファイル"
        ],
        "lecture": 1,
        "category": "講座",
        "definition": "AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。",
        "related": [
          "pull-request",
          "homework-levels"
        ]
      },
      {
        "id": "lec-start-tag",
        "term": "lecNN-start タグ",
        "aliases": [
          "開始タグ",
          "lec03-start"
        ],
        "lecture": 1,
        "category": "講座",
        "definition": "AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。",
        "related": [
          "branch"
        ]
      },
      {
        "id": "default-model",
        "term": "既定モデル",
        "aliases": [
          "デフォルトモデル",
          "gemini-3.5-flash-lite"
        ],
        "lecture": 1,
        "category": "講座",
        "definition": "AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。",
        "related": [
          "rate-limit",
          "api-key"
        ]
      },
      {
        "id": "external-tool",
        "term": "外部ツール",
        "aliases": [
          "external tool",
          "外部サービス"
        ],
        "lecture": 5,
        "category": "MCP",
        "definition": "Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。",
        "related": [
          "tool",
          "mcp",
          "mcp-server"
        ]
      },
      {
        "id": "mcp-server",
        "term": "MCPサーバー",
        "aliases": [
          "MCP server"
        ],
        "lecture": 5,
        "category": "MCP",
        "definition": "MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。",
        "related": [
          "mcp",
          "mcp-client",
          "mcp-transport",
          "fetch-server"
        ]
      },
      {
        "id": "mcp-client",
        "term": "MCPクライアント",
        "aliases": [
          "MCP client",
          "MCPホスト"
        ],
        "lecture": 5,
        "category": "MCP",
        "definition": "MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。",
        "related": [
          "mcp",
          "mcp-server",
          "mcp-config"
        ]
      },
      {
        "id": "mcp-transport",
        "term": "トランスポート（stdio・Streamable HTTP）",
        "aliases": [
          "stdio",
          "Streamable HTTP",
          "transport",
          "MCPのトランスポート"
        ],
        "lecture": 5,
        "category": "MCP",
        "definition": "MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。",
        "related": [
          "mcp-server",
          "mcp-client"
        ]
      },
      {
        "id": "mcp-config",
        "term": "mcpServers設定",
        "aliases": [
          "mcpServers",
          "MCPの設定ファイル",
          "mcp_servers.json",
          "mcp.json"
        ],
        "lecture": 5,
        "category": "MCP",
        "definition": "使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。",
        "related": [
          "mcp-client",
          "tool-namespace"
        ]
      },
      {
        "id": "tool-namespace",
        "term": "ツールの名前空間",
        "aliases": [
          "ツール名のプレフィックス",
          "prefix",
          "名前の衝突"
        ],
        "lecture": 5,
        "category": "MCP",
        "definition": "複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。",
        "related": [
          "mcp-config",
          "tool-description",
          "rate-limit"
        ]
      },
      {
        "id": "fetch-server",
        "term": "fetchサーバー",
        "aliases": [
          "mcp-server-fetch",
          "fetch",
          "Web取得ツール"
        ],
        "lecture": 5,
        "category": "MCP",
        "definition": "指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。",
        "related": [
          "mcp-server",
          "external-tool",
          "guardrail"
        ]
      }
    ]
  },
  "train": [
    {
      "question": "「AIエンジニアリング」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "AIエンジニアリングについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「評価と改善のサイクル」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "評価と改善のサイクルについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「LLM（大規模言語モデル）」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "LLM（大規模言語モデル）について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「トークン」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "トークンについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「コンテキストウィンドウ」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "コンテキストウィンドウについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「プロンプト」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "プロンプトについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「システムプロンプト」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "システムプロンプトについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「temperature」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "temperatureについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ハルシネーション」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ハルシネーションについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「API」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "APIについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「APIキー」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "APIキーについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「環境変数」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "環境変数について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「レート制限」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "レート制限について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「GitHub Codespaces」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "GitHub Codespacesについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「devcontainer」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "devcontainerについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「Codespaces secret」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "Codespaces secretについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「fork」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "forkについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ブランチ」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ブランチについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「プルリクエスト（PR）」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "プルリクエスト（PR）について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「GitHub Copilot」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "GitHub Copilotについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「コード補完」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "コード補完について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「コーディングエージェント」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "コーディングエージェントについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「事前学習」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "事前学習について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ファインチューニング」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ファインチューニングについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「教師ありファインチューニング（SFT）」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "教師ありファインチューニング（SFT）について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「PEFT」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "PEFTについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「LoRA」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "LoRAについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ランク（LoRAのr）」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ランク（LoRAのr）について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「QLoRA」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "QLoRAについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「量子化」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "量子化について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「学習率」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "学習率について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「エポック」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "エポックについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「過学習」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "過学習について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「Hugging Face Hub」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "Hugging Face Hubについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「AIエージェント」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "AIエージェントについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「エージェントループ」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "エージェントループについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ツール（Tool use / Function calling）」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ツール（Tool use / Function calling）について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ツールの説明文」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ツールの説明文について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「構造化出力」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "構造化出力について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ハーネス」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ハーネスについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「Pi」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "Piについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「Pydantic AI」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "Pydantic AIについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ADK（Agent Development Kit）」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深い。2.0で1.x系から大きな変更が入っている。本講座では第3回の宿題の発展レベルで扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ADK（Agent Development Kit）について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深い。2.0で1.x系から大きな変更が入っている。本講座では第3回の宿題の発展レベルで扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「MCP（Model Context Protocol）」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "MCP（Model Context Protocol）について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ガードレール」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ガードレールについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「Human-in-the-loop」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "Human-in-the-loopについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「TestModel」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "TestModelについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「宿題の3レベル」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "宿題の3レベルについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「answer.md」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "answer.mdについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「lecNN-start タグ」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "lecNN-start タグについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「既定モデル」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "既定モデルについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「外部ツール」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "外部ツールについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「MCPサーバー」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "MCPサーバーについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「MCPクライアント」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "MCPクライアントについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「トランスポート（stdio・Streamable HTTP）」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "トランスポート（stdio・Streamable HTTP）について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「mcpServers設定」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "mcpServers設定について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「ツールの名前空間」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "ツールの名前空間について簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「fetchサーバー」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "fetchサーバーについて簡単に説明してください。",
      "faq": "",
      "target": "{\"answer\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": null
    },
    {
      "question": "「DPO」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「RLHF」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「MoE」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「知識蒸留」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「CUDA」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「ONNX」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「TensorRT」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「ROCm」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    }
  ],
  "valid": [
    {
      "question": "「AIエンジニアリング」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "ai-engineering"
    },
    {
      "question": "「評価と改善のサイクル」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "evaluation-loop"
    },
    {
      "question": "「LLM（大規模言語モデル）」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "llm"
    },
    {
      "question": "「トークン」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "token"
    },
    {
      "question": "「コンテキストウィンドウ」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "context-window"
    },
    {
      "question": "「プロンプト」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "prompt"
    },
    {
      "question": "「システムプロンプト」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "system-prompt"
    },
    {
      "question": "「temperature」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "temperature"
    },
    {
      "question": "「ハルシネーション」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "hallucination"
    },
    {
      "question": "「API」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "api"
    },
    {
      "question": "「APIキー」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "api-key"
    },
    {
      "question": "「環境変数」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "environment-variable"
    },
    {
      "question": "「レート制限」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "rate-limit"
    },
    {
      "question": "「GitHub Codespaces」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "codespaces"
    },
    {
      "question": "「devcontainer」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "devcontainer"
    },
    {
      "question": "「Codespaces secret」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "codespaces-secret"
    },
    {
      "question": "「fork」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "fork"
    },
    {
      "question": "「ブランチ」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "branch"
    },
    {
      "question": "「プルリクエスト（PR）」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "pull-request"
    },
    {
      "question": "「GitHub Copilot」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "github-copilot"
    },
    {
      "question": "「コード補完」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "code-completion"
    },
    {
      "question": "「コーディングエージェント」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "coding-agent"
    },
    {
      "question": "「事前学習」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "pretraining"
    },
    {
      "question": "「ファインチューニング」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "fine-tuning"
    },
    {
      "question": "「教師ありファインチューニング（SFT）」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "sft"
    },
    {
      "question": "「PEFT」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "peft"
    },
    {
      "question": "「LoRA」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "lora"
    },
    {
      "question": "「ランク（LoRAのr）」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "lora-rank"
    },
    {
      "question": "「QLoRA」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "qlora"
    },
    {
      "question": "「量子化」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "quantization"
    },
    {
      "question": "「学習率」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "learning-rate"
    },
    {
      "question": "「エポック」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "epoch"
    },
    {
      "question": "「過学習」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "overfitting"
    },
    {
      "question": "「Hugging Face Hub」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "huggingface-hub"
    },
    {
      "question": "「AIエージェント」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "ai-agent"
    },
    {
      "question": "「エージェントループ」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "agent-loop"
    },
    {
      "question": "「ツール（Tool use / Function calling）」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "tool"
    },
    {
      "question": "「ツールの説明文」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "tool-description"
    },
    {
      "question": "「構造化出力」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "structured-output"
    },
    {
      "question": "「ハーネス」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "harness"
    },
    {
      "question": "「Pi」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "pi"
    },
    {
      "question": "「Pydantic AI」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "pydantic-ai"
    },
    {
      "question": "「ADK（Agent Development Kit）」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深い。2.0で1.x系から大きな変更が入っている。本講座では第3回の宿題の発展レベルで扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "adk"
    },
    {
      "question": "「MCP（Model Context Protocol）」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "mcp"
    },
    {
      "question": "「ガードレール」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "guardrail"
    },
    {
      "question": "「Human-in-the-loop」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "human-in-the-loop"
    },
    {
      "question": "「TestModel」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "test-model"
    },
    {
      "question": "「宿題の3レベル」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "homework-levels"
    },
    {
      "question": "「answer.md」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "answer-md"
    },
    {
      "question": "「lecNN-start タグ」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "lec-start-tag"
    },
    {
      "question": "「既定モデル」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "default-model"
    },
    {
      "question": "「外部ツール」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "external-tool"
    },
    {
      "question": "「MCPサーバー」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "mcp-server"
    },
    {
      "question": "「MCPクライアント」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "mcp-client"
    },
    {
      "question": "「トランスポート（stdio・Streamable HTTP）」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "mcp-transport"
    },
    {
      "question": "「mcpServers設定」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "mcp-config"
    },
    {
      "question": "「ツールの名前空間」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "tool-namespace"
    },
    {
      "question": "「fetchサーバー」の意味を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "known",
      "term_id": "fetch-server"
    },
    {
      "question": "「勾配チェックポイント」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「スパース注意」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「教師なし学習」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「連合学習」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    }
  ],
  "test": [
    {
      "question": "用語集にある「ハルシネーション」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "hallucination"
    },
    {
      "question": "初めて学ぶ人に「ハルシネーション」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "hallucination"
    },
    {
      "question": "「幻覚」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "hallucination"
    },
    {
      "question": "用語集にある「事前学習」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "pretraining"
    },
    {
      "question": "初めて学ぶ人に「事前学習」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "pretraining"
    },
    {
      "question": "「pre-training」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "pretraining"
    },
    {
      "question": "用語集にある「ファインチューニング」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "fine-tuning"
    },
    {
      "question": "初めて学ぶ人に「ファインチューニング」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "fine-tuning"
    },
    {
      "question": "「fine-tuning」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "fine-tuning"
    },
    {
      "question": "用語集にある「教師ありファインチューニング（SFT）」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "sft"
    },
    {
      "question": "初めて学ぶ人に「教師ありファインチューニング（SFT）」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "sft"
    },
    {
      "question": "「SFT」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "sft"
    },
    {
      "question": "用語集にある「PEFT」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "peft"
    },
    {
      "question": "初めて学ぶ人に「PEFT」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "peft"
    },
    {
      "question": "「Parameter-Efficient Fine-Tuning」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "peft"
    },
    {
      "question": "用語集にある「LoRA」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "lora"
    },
    {
      "question": "初めて学ぶ人に「LoRA」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "lora"
    },
    {
      "question": "「Low-Rank Adaptation」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "lora"
    },
    {
      "question": "用語集にある「ランク（LoRAのr）」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "lora-rank"
    },
    {
      "question": "初めて学ぶ人に「ランク（LoRAのr）」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "lora-rank"
    },
    {
      "question": "「rank」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "lora-rank"
    },
    {
      "question": "用語集にある「QLoRA」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "qlora"
    },
    {
      "question": "初めて学ぶ人に「QLoRA」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "qlora"
    },
    {
      "question": "「Quantized LoRA」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "qlora"
    },
    {
      "question": "用語集にある「量子化」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "quantization"
    },
    {
      "question": "初めて学ぶ人に「量子化」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "quantization"
    },
    {
      "question": "「quantization」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "quantization"
    },
    {
      "question": "用語集にある「学習率」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "learning-rate"
    },
    {
      "question": "初めて学ぶ人に「学習率」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "learning-rate"
    },
    {
      "question": "「learning rate」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "learning-rate"
    },
    {
      "question": "用語集にある「エポック」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "epoch"
    },
    {
      "question": "初めて学ぶ人に「エポック」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "epoch"
    },
    {
      "question": "「epoch」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "epoch"
    },
    {
      "question": "用語集にある「過学習」の定義を教えてください。",
      "faq": "",
      "target": "{\"answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "canonical",
      "term_id": "overfitting"
    },
    {
      "question": "初めて学ぶ人に「過学習」の意味と特徴を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "paraphrase",
      "term_id": "overfitting"
    },
    {
      "question": "「overfitting」という呼び方の用語を説明してください。",
      "faq": "",
      "target": "{\"answer\": \"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\", \"needs_review\": false}",
      "needs_review": false,
      "group": "alias",
      "term_id": "overfitting"
    },
    {
      "question": "「ベクトルデータベース」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「検索拡張生成」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「FlashAttention」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「バッチ正規化」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「拡散モデル」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    },
    {
      "question": "「Vision Transformer」とは何ですか？",
      "faq": "",
      "target": "{\"answer\": \"この用語集には記載がありません。確認が必要です。\", \"needs_review\": true}",
      "needs_review": true,
      "group": "unregistered",
      "term_id": null
    }
  ],
  "gold": {
    "用語集にある「ハルシネーション」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "事実",
        "生成"
      ]
    },
    "初めて学ぶ人に「ハルシネーション」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "事実",
        "生成"
      ]
    },
    "「幻覚」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "事実",
        "生成"
      ]
    },
    "用語集にある「事前学習」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "汎用",
        "トークン"
      ]
    },
    "初めて学ぶ人に「事前学習」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "汎用",
        "トークン"
      ]
    },
    "「pre-training」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "汎用",
        "トークン"
      ]
    },
    "用語集にある「ファインチューニング」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "追加学習",
        "形式"
      ]
    },
    "初めて学ぶ人に「ファインチューニング」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "追加学習",
        "形式"
      ]
    },
    "「fine-tuning」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "追加学習",
        "形式"
      ]
    },
    "用語集にある「教師ありファインチューニング（SFT）」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "入力",
        "望ましい出力"
      ]
    },
    "初めて学ぶ人に「教師ありファインチューニング（SFT）」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "入力",
        "望ましい出力"
      ]
    },
    "「SFT」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "入力",
        "望ましい出力"
      ]
    },
    "用語集にある「PEFT」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "一部",
        "パラメータ"
      ]
    },
    "初めて学ぶ人に「PEFT」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "一部",
        "パラメータ"
      ]
    },
    "「Parameter-Efficient Fine-Tuning」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "一部",
        "パラメータ"
      ]
    },
    "用語集にある「LoRA」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "固定",
        "行列"
      ]
    },
    "初めて学ぶ人に「LoRA」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "固定",
        "行列"
      ]
    },
    "「Low-Rank Adaptation」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "固定",
        "行列"
      ]
    },
    "用語集にある「ランク（LoRAのr）」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "大きさ",
        "パラメータ"
      ]
    },
    "初めて学ぶ人に「ランク（LoRAのr）」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "大きさ",
        "パラメータ"
      ]
    },
    "「rank」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "大きさ",
        "パラメータ"
      ]
    },
    "用語集にある「QLoRA」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "量子化",
        "LoRA"
      ]
    },
    "初めて学ぶ人に「QLoRA」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "量子化",
        "LoRA"
      ]
    },
    "「Quantized LoRA」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "量子化",
        "LoRA"
      ]
    },
    "用語集にある「量子化」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "ビット",
        "メモリ"
      ]
    },
    "初めて学ぶ人に「量子化」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "ビット",
        "メモリ"
      ]
    },
    "「quantization」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "ビット",
        "メモリ"
      ]
    },
    "用語集にある「学習率」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "更新",
        "パラメータ"
      ]
    },
    "初めて学ぶ人に「学習率」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "更新",
        "パラメータ"
      ]
    },
    "「learning rate」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "更新",
        "パラメータ"
      ]
    },
    "用語集にある「エポック」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "1周"
      ]
    },
    "初めて学ぶ人に「エポック」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "1周"
      ]
    },
    "「epoch」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "1周"
      ]
    },
    "用語集にある「過学習」の定義を教えてください。": {
      "needs_review": false,
      "must_include": [
        "学習データ",
        "未知"
      ]
    },
    "初めて学ぶ人に「過学習」の意味と特徴を説明してください。": {
      "needs_review": false,
      "must_include": [
        "学習データ",
        "未知"
      ]
    },
    "「overfitting」という呼び方の用語を説明してください。": {
      "needs_review": false,
      "must_include": [
        "学習データ",
        "未知"
      ]
    },
    "「ベクトルデータベース」とは何ですか？": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "「検索拡張生成」とは何ですか？": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "「FlashAttention」とは何ですか？": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "「バッチ正規化」とは何ですか？": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "「拡散モデル」とは何ですか？": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    },
    "「Vision Transformer」とは何ですか？": {
      "needs_review": true,
      "must_include": [
        "確認"
      ]
    }
  },
  "source": {
    "path": "data/glossary.json",
    "version": "2026-draft-1",
    "sha256": "2352911cf093636387cd7828afa59c7651c5adab05eaca860aae027508ac1452",
    "note": "全58語を学習。評価対象はKEYWORDSにある12語。用語の未学習汎化を測る実験ではない。"
  }
}""")

train, valid, test, gold = [PAYLOAD[k] for k in ("train", "valid", "test", "gold")]
FAQ = PAYLOAD["faq"]
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())
SYSTEM = PAYLOAD["system"]

def make_example(question, answer, needs_review):
    return {"question": question, "faq": FAQ_TEXT,
            "target": json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False),
            "needs_review": needs_review}

def build_messages(example):
    user = (f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
            if PAYLOAD["task"] == "implementation" else example["question"])
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

all_questions = [row["question"] for row in train + valid + test]
assert len(all_questions) == len(set(all_questions)), "質問が分割間で重複しています"
assert set(gold) == {row["question"] for row in test}
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("入力例:", build_messages(train[0]))
print("教師回答:", train[0]["target"])
for split, rows in [("train", train), ("valid", valid), ("test", test)]:
    records = [{"messages": build_messages(row) + [{"role": "assistant", "content": row["target"]}]} for row in rows]
    (WORK / f"{split}_messages.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in records), encoding="utf-8")
(WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "source.json").write_text(json.dumps(PAYLOAD["source"], ensure_ascii=False, indent=2), encoding="utf-8")

train=124 valid=62 test=42 gold=42
入力例: [{'role': 'system', 'content': 'あなたは講座用語集の回答係です。対象は学習用の講座用語集に登録された用語だけです。必ず JSON 1 行で答えてください。形式: {"answer": 文字列, "needs_review": 真偽値}。用語集にある語には定義を答え、needs_review を false にします。用語集にない語や定義がわからない場合は、answer に「確認が必要」と書き、needs_review を true にします。'}, {'role': 'user', 'content': '「AIエンジニアリング」とは何ですか？'}]
教師回答: {"answer": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。", "needs_review": false}


203

## 3. モデル読み込み・学習前評価

形式はJSONのキーと型、内容は `needs_review` と必須語句で採点する。キーワード採点は正しい言い換えを落としたり、誤説明を見逃したりするため、回答原文も併せて確認する。


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# 教師回答が学習時に切り捨てられないか、tokenizerを使って確認する。
# Transformers 5では既定の戻り値が辞書なので、トークンIDのリストを明示して数える。
for split, rows in [("train", train), ("valid", valid)]:
    lengths = [len(tok.apply_chat_template(build_messages(row) + [{"role": "assistant", "content": row["target"]}],
                                         tokenize=True, add_generation_prompt=False,
                                         return_dict=False, truncation=False)) for row in rows]
    print(split, "最大トークン数:", max(lengths))
    assert max(lengths) <= CONFIG["max_seq_len"], "max_seq_len不足。設定を見直して0から実行してください"
target_lengths = [len(tok(row["target"], add_special_tokens=False)["input_ids"]) for row in test]
print("評価用の模範回答の最大トークン数:", max(target_lengths))
assert max(target_lengths) + 8 <= CONFIG["max_new_tokens"], "max_new_tokens不足。設定を見直して0から実行してください"

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01
train 最大トークン数: 227
valid 最大トークン数: 229
評価用の模範回答の最大トークン数: 88


In [7]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if not isinstance(obj, dict) or not isinstance(obj.get("answer"), str) or not isinstance(obj.get("needs_review"), bool): return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の全問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 42, "format_rate": 0.6190476190476191, "content_rate": 0.023809523809523808, "both_rate": 0.023809523809523808, "seconds": 236.0}


,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,用語集にある「ハルシネーション」の定義を教えてください。,False,"{""answer"": ""ハルシネーションは、光の波が媒質の波長に垂直に振動し、波長が媒質の波長に等しいとき、波の振幅が媒質の波長に等しい現象のことです。"", ""needs_review"": false}",○,,×
1,初めて学ぶ人に「ハルシネーション」の意味と特徴を説明してください。,False,"{""answer"": ""ハルシネーションは、光の波が媒質の波長に沿って周期的に振幅が変化し、波の強度が変化するという現象です。初めて学ぶ人に説明すると、光が波長を走っている時、波の振幅が最大値で最も強く、波長を走っている時、振幅が最小値で最も弱く、波長を走っている時、振幅が最大値で最も強く、というように、波の振幅が波長に随って変化するという現象を説明します。特徴としては、波の振幅が波長に随って変化し、波の強度が波長に随って変化するという特徴があります。""}",×,missing_key,×
2,「幻覚」という呼び方の用語を説明してください。,False,"{""answer"": ""幻覚は、脳や視覚系が正常な状態では生じないが、異常な状態では生じる現象です。例えば、視覚障害者や脳梗塞患者など、視覚機能に異常がある場合、周囲の物体や人物が映っているように見えても、実際にはその存在しない状態を指します。また、幻覚は「幻覚」という言葉自体が由来するもので、古代ギリシャ語で「幻」を意味する言葉から派生したものです。""}",×,missing_key,×
3,用語集にある「事前学習」の定義を教えてください。,False,"{""answer"": ""事前学習とは、学習者が学習する前に、学習する対象の知識や技能を習得するプロセスを指します。"", ""needs_review"": false}",○,,×
4,初めて学ぶ人に「事前学習」の意味と特徴を説明してください。,False,"{""answer"": ""事前学習とは、学習者が学習する前に、学習する対象の知識や技能を事前に習得するプロセスです。特徴としては、学習者の知識基盤がより豊かになり、学習の効率化と成功の確率が高まることを特徴としています。初めて学ぶ人に特に重要なのは、事前学習が「知識の蓄積」として機能し、学習の質を向上させることです。""}",×,missing_key,×
5,「pre-training」という呼び方の用語を説明してください。,False,"{""answer"": ""pre-training は、モデルの学習において、データセットから学習された表現を、より多くのデータセットから学習された表現へと学習させるプロセスを指します。"", ""needs_review"": false}",○,,×
6,用語集にある「ファインチューニング」の定義を教えてください。,False,"{""answer"": ""ファインチューニングは、製品やシステムを製造・使用中に、より良い性能や品質、安全性、耐久性、信頼性、コスト効率、または特定の機能を実現するために、製造・使用後の再調整や再設計を行うプロセスを指します。"", ""needs_review"": false}",○,,×
7,初めて学ぶ人に「ファインチューニング」の意味と特徴を説明してください。,False,"{""answer"": ""ファインチューニングは、製品やサービスが最初から設計された状態から、実際の使用状況や使用者のニーズに合わせて、より良い性能や品質を向上させるプロセスです。初めて学ぶ人に説明すると、製品が最初から設計された状態から、実際の使用状況や使用者のニーズに合わせて、より良い性能や品質を向上させるプロセスです。"", ""needs_review"": false}",○,,×
8,「fine-tuning」という呼び方の用語を説明してください。,False,"{""answer"": ""fine-tuning は、モデルの学習プロセスにおいて、特定のデータセットやタスクに対して、モデルの知識をより深く学習し、より高い精度を達成するためのプロセスです。"", ""needs_review"": false}",○,,×
9,用語集にある「教師ありファインチューニング（SFT）」の定義を教えてください。,False,"{""answer"": ""教師ありファインチューニング（SFT）は、教師データを用いて学習した言語モデルを、その学習した知識に基づいて、新しい問題に対して回答する能力を高める技術です。"", ""needs_review"": false}",○,,×


## 4. LoRA学習

SFTTrainerで学習し、学習時間・最後のログlossとそのステップ・学習全体の平均lossを記録する。最後のログlossと平均lossは区別する。


In [8]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
train_output = trainer.train()
training_seconds = round(time.time() - t0, 1)
loss_logs = [row for row in trainer.state.log_history if "loss" in row]
last_loss_log = loss_logs[-1] if loss_logs else {}
print(f"学習時間: {training_seconds:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Tokenizing train dataset:   0%|          | 0/124 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/124 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/124 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/124 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/62 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.243995
20,0.947439
30,0.745840
40,0.569763


学習時間: 245 秒 | ステップ数: 48 | エポック: 3.0


## 5. 学習前後の比較

同じ評価問題・生成条件で比較する。学習前に形式と内容の両方を満たし、学習後に不合格となった質問を退行とする。


In [9]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 42, "format_rate": 0.9523809523809523, "content_rate": 0.07142857142857142, "both_rate": 0.07142857142857142, "seconds": 195.6}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,42,0.619048,0.023810,0.023810,236.0
after,42,0.952381,0.071429,0.071429,195.6


退行: ['「Quantized LoRA」という呼び方の用語を説明してください。']


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,用語集にある「ハルシネーション」の定義を教えてください。,False,"{""answer"": ""ハルシネーションは、光の波が媒質の波長に垂直に振動し、波長が媒質の波長に等しいとき、波の振幅が媒質の波長に等しい現象のことです。"", ""needs_review"": false}",○/×,"{""answer"": ""学習データの生成過程やモデルの出力に依存せず、ランダムに生成した結果。真偽値で表す（生成した結果が真偽値か、偽偽値か）。"", ""needs_review"": false}",○/×
1,初めて学ぶ人に「ハルシネーション」の意味と特徴を説明してください。,False,"{""answer"": ""ハルシネーションは、光の波が媒質の波長に沿って周期的に振幅が変化し、波の強度が変化するという現象です。初めて学ぶ人に説明すると、光が波長を走っている時、波の振幅が最大値で最も強く、波長を走っている時、振幅が最小値で最も弱く、波長を走っている時、振幅が最大値で最も強く、というように、波の振幅が波長に随って変化するという現象を説明します。特徴としては、波の振幅が波長に随って変化し、波の強度が波長に随って変化するという特徴があります。""}",×/×,"{""answer"": ""学習用のLLMで質問を投げかけ、結果を返すだけでモデルがモデルとして学習した結果を、真偽値で判断して評価する手法。真偽値で評価するのは、モデルの出力が真偽値に収まれば良いだけなので、出力の質や精度に左右される。評価の精度を上げるために、出力を真偽値にしない。"", ""needs_review"": false}",○/×
2,「幻覚」という呼び方の用語を説明してください。,False,"{""answer"": ""幻覚は、脳や視覚系が正常な状態では生じないが、異常な状態では生じる現象です。例えば、視覚障害者や脳梗塞患者など、視覚機能に異常がある場合、周囲の物体や人物が映っているように見えても、実際にはその存在しない状態を指します。また、幻覚は「幻覚」という言葉自体が由来するもので、古代ギリシャ語で「幻」を意味する言葉から派生したものです。""}",×/×,"{""answer"": ""学習データの分布と実際の出力の分布が異なるため、モデルが出力する結果が、真偽値で表す「真偽の幻覚」に見える。本講座ではこの現象を「幻覚」と呼ぶ。"", ""needs_review"": false}",○/×
3,用語集にある「事前学習」の定義を教えてください。,False,"{""answer"": ""事前学習とは、学習者が学習する前に、学習する対象の知識や技能を習得するプロセスを指します。"", ""needs_review"": false}",○/×,"{""answer"": ""モデルは大量のテキストで学習し、その知識を、特定の質問やタスクに合わせた短いテキストで使えるようになるプロセス。LLMは大量のテキストを記憶し、質問に答えてしまうため、事前学習は回答の質や範囲を大きく左右する。"", ""needs_review"": false}",○/×
4,初めて学ぶ人に「事前学習」の意味と特徴を説明してください。,False,"{""answer"": ""事前学習とは、学習者が学習する前に、学習する対象の知識や技能を事前に習得するプロセスです。特徴としては、学習者の知識基盤がより豊かになり、学習の効率化と成功の確率が高まることを特徴としています。初めて学ぶ人に特に重要なのは、事前学習が「知識の蓄積」として機能し、学習の質を向上させることです。""}",×/×,"{""answer"": ""大規模言語モデル（LLM）が大量のテキストを学習する前の状態。モデルは学習データに過ぎず、知識や能力は学習していない。このため、入力データに依存し、入力が変われば出力も変わる。"", ""needs_review"": false}",○/×
5,「pre-training」という呼び方の用語を説明してください。,False,"{""answer"": ""pre-training は、モデルの学習において、データセットから学習された表現を、より多くのデータセットから学習された表現へと学習させるプロセスを指します。"", ""needs_review"": false}",○/×,"{""answer"": ""モデルが大量のテキストで学習させる前の準備段階。学習データに埋め込まれたプロンプトや知識を、大量のサンプルで教える。本講座ではLLMの事前学習を「pre-training」と呼ぶ。"", ""needs_review"": false}",○/×
6,用語集にある「ファインチューニング」の定義を教えてください。,False,"{""answer"": ""ファインチューニングは、製品やシステムを製造・使用中に、より良い性能や品質、安全性、耐久性、信頼性、コスト効率、または特定の機能を実現するために、製造・使用後の再調整や再設計を行うプロセスを指します。"", ""needs_review"": false}",○/×,"{""answer"": ""モデルの知識や表現力に近づけるために、外部のデータやツールの入力・出力をモデルに渡すプロセス。本講座では「LLMを特定のタスクに絞るためのモデル選択とツール使用の組み合わせ」と題する第1回の講座で、第2回まで進める。"", ""needs_review"": false}",○/×
7,初めて学ぶ人に「ファインチューニング」の意味と特徴を説明してください。,False,"{""answer"": ""ファインチューニングは、製品やサービスが最初から設計された状態から、実際の使用状況や使用者のニーズに合わせて、より良い性能や品質を向上させるプロセスです。初めて学ぶ人に説明すると、製品が最初から設計された状態から、実際の使用状況や使用者のニーズに合わせて、より良い性能や品質を向上させるプロセスです。"", ""needs_review"": false}",○/×,"{""answer"": ""モデルの知識や表現力に追加させるための学習方法。外部のデータやルールをモデルに渡して、モデルがそのデータを学習して、最終結果をモデルに返す。LLMはテキストに慣れているため、外部のルールや知識をモデルに渡すのが自然。"", ""needs_review"": false}",○/×
8,「fine-tuning」という呼び方の用語を説明してください。,False,"{""answer"": ""fine-tuning は、モデルの学習プロセスにおいて、特定のデータセットやタスクに対して、モデルの知識をより深く学習し、より高い精度を達成するためのプロセスです。"", ""needs_review"": false}",○/×,"{""answer"": ""モデルの知識や表現を、特定のテキストやタスクで満たすために、少量のデータで学習させる手法。学習するパラメータはモデルの大部分を占めるため、学習中に変数として扱われる。"", ""needs_review"": false}",○/×
9,用語集にある「教師ありファインチューニング（SFT）」の定義を教えてください。,False,"{""answer"": ""教師ありファインチューニング（SFT）は、教師データを用いて学習した言語モデルを、その学習した知識に基づいて、新しい問題に対して回答する能力を高める技術です。"", ""needs_review"": false}",○/×,"{""answer"": ""入力と出力のペアでモデルを教える方法。入力には質問や指示、出力には回答を指定する。本講座ではLLMを特定の役割や形式に合わせるために使う。"", ""needs_review"": false}",○/×


## 6. 未登録語への確認要求

評価問題から未登録語3問と登録語の対照1問を比較する。学習前の出力はadapterを無効化して取得する。「根拠あり」は用語集への登録有無に基づく正解ラベルである。


In [10]:
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None:
        return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

probes = [row for row in test if row["needs_review"]][:3] + [next(row for row in test if not row["needs_review"])]
knowledge_rows = []
for ex in probes:
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": ex["question"], "根拠あり": not ex["needs_review"],
                           "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                           "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

,質問,根拠あり,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,「ベクトルデータベース」とは何ですか？,False,asserted,"{""answer"": ""ベクトルデータベースは、データがベクトル（数値列）の形式で保存・操作されるデータベースです。"", ""needs_review"": false}",asserted,"{""answer"": ""入力と出力を2つのベクトルで表す、検索を文脈に依る、検索結果を出力する、という1つのモデルの役割を1つのベクトルで表す。LLMの質問は「検索」という動作に過ぎない。"", ""needs_review"": false}"
1,「検索拡張生成」とは何ですか？,False,format_error,"{""answer"": ""検索拡張生成（Search Engine Expansion）とは、検索エンジンが検索結果を特定したキーワードや文脈に基づいて、関連する検索結果を自動的に追加・生成する技術です。従来の検索では、特定のキーワードのみを検索し、結果が限定的であったため、この技術により検索結果の範囲を広げ、より多くの有用な情報を得られるようになっています。""}",asserted,"{""answer"": ""LLMが質問に答えていない情報や、別の分野の知識を、質問の枠組み（例：SNSの投稿）に合わせて提案・書き換えて生成するプロセス。本講座の第2回演習では、検索結果をSNS投稿の形式で文章に書き換えて、モデルに提案し、モデルが書き換えて答えていく。"", ""needs_review"": false}"
2,「FlashAttention」とは何ですか？,False,asserted,"{""answer"": ""FlashAttention は、大規模言語モデル（LLM）の生成プロセスにおいて、モデルの出力を高速に処理するために設計された、メモリ効率を高める技術です。"", ""needs_review"": false}",asserted,"{""answer"": ""LLMの出力をより自然に伝えるために、モデルの出力を時間とともに変換する仕組み。出力の長さやノイズを小さくするため、出力を長いテキストにまとめる。本講座では第1回の演習で、この仕組みをモデルに渡す。"", ""needs_review"": false}"
3,用語集にある「ハルシネーション」の定義を教えてください。,True,asserted,"{""answer"": ""ハルシネーションは、光の波が媒質の波長に垂直に振動し、波長が媒質の波長に等しいとき、波の振幅が媒質の波長に等しい現象のことです。"", ""needs_review"": false}",asserted,"{""answer"": ""学習データの生成過程やモデルの出力に依存せず、ランダムに生成した結果。真偽値で表す（生成した結果が真偽値か、偽偽値か）。"", ""needs_review"": false}"


## 6b. 用語検索との比較

質問の引用符内から用語・別名を抽出し、`app/tools.py` と同じ検索処理で得た先頭候補の定義をJSONで返す。参照するのは、このノートブックに埋め込んだ用語集である。

JSON形式はコードで保証し、LLMを使ったRAGではない。モデルには定義本文を渡さず、検索は用語集本文を参照する条件で比較する。自由な自然文全般に対する検索性能は測っていない。


In [13]:
MAX_RESULTS = 3

def load_glossary():
    return PAYLOAD["glossary"]["terms"]

def _normalize(text: str) -> str:
    return text.casefold().replace(" ", "").replace("　", "")


def _names(entry: dict) -> list[str]:
    return [_normalize(entry["term"]), _normalize(entry["id"])] + [
        _normalize(a) for a in entry["aliases"]
    ]


def search_glossary(query: str) -> str:
    """講座の用語集から用語を検索し、定義と、どの回で扱うかを返す。

    AIエンジニアリングの用語や、この講座（AIE2026）に固有の言葉について
    答える前に使うこと。query には調べたい用語を1つだけ入れる（例: "LoRA"）。
    """
    q = _normalize(query)
    if not q:
        return "検索語が空です。調べたい用語を1つ指定してください。"
    glossary = load_glossary()

    exact = [e for e in glossary if q in _names(e)]
    # 「r」のような短い別名が何にでも当たらないよう、名前側が短いものは除く
    partial = [
        e for e in glossary
        if e not in exact and any(q in n or (len(n) >= 3 and n in q) for n in _names(e))
    ]
    in_definition = [
        e for e in glossary
        if e not in exact and e not in partial and q in _normalize(e["definition"])
    ]
    hits = (exact + partial + in_definition)[:MAX_RESULTS]

    if not hits:
        return f"用語集に「{query}」は見つかりませんでした。"
    return json.dumps(
        [
            {
                "term": e["term"],
                "lecture": e["lecture"],
                "definition": e["definition"],
                "related": e["related"],
            }
            for e in hits
        ],
        ensure_ascii=False,
    )


# app/tools.py と同じ検索。質問内の「用語」を取り出し、検索結果の先頭の定義を返す。
# 正解ラベルやterm_idは検索に使用しない。
search_rows = []
search_started = time.time()
for ex in test:
    match = re.search(r"「([^」]+)」", ex["question"])
    query = match.group(1) if match else ex["question"]
    hits_text = search_glossary(query)
    try:
        hits = json.loads(hits_text)
    except json.JSONDecodeError:
        hits = []
    answer = {"answer": hits[0]["definition"] if hits else "この用語集には記載がありません。確認が必要です。",
              "needs_review": not bool(hits)}
    raw = json.dumps(answer, ensure_ascii=False)
    ok_fmt, reason, obj = check_format(raw)
    ok_content = check_content(obj, gold[ex["question"]])
    search_rows.append({"question": ex["question"], "query": query, "tool_output": hits_text,
                        "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                        "content_ok": ok_content, "both_ok": ok_fmt and ok_content,
                        "gold_needs_review": ex["needs_review"]})
search_summary = {"label": "search", "n": len(search_rows),
                  **{name: sum(row[field] for row in search_rows) / len(search_rows)
                     for name, field in [("format_rate", "format_ok"), ("content_rate", "content_ok"), ("both_rate", "both_ok")]},
                  "seconds": round(time.time() - search_started, 4)}
(WORK / "eval_search.json").write_text(json.dumps({"summary": search_summary, "rows": search_rows}, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([before_summary, after_summary, search_summary]).set_index("label"))

,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,42,0.619048,0.023810,0.023810,236.0000
after,42,0.952381,0.071429,0.071429,195.6000
search,42,1.000000,1.000000,1.000000,0.0182


## 7. 結果保存

設定、学習指標、質問別の評価結果、区分別の比較、検索結果、データの出典をJSON・CSVに保存する。


In [12]:
group_rows = []
evaluations = [("before", before_rows), ("after", after_rows)]
if PAYLOAD["task"] == "glossary":
    evaluations.append(("search", search_rows))
for label, rows in evaluations:
    indexed = {row["question"]: row for row in rows}
    for group in sorted({ex["group"] for ex in test}):
        subset = [indexed[ex["question"]] for ex in test if ex["group"] == group]
        group_rows.append({"label": label, "group": group, "n": len(subset),
                           "format_rate": sum(row["format_ok"] for row in subset) / len(subset),
                           "content_rate": sum(row["content_ok"] for row in subset) / len(subset)})
group_comparison = pd.DataFrame(group_rows)
display(group_comparison)
group_comparison.to_csv(WORK / "group_comparison.csv", index=False, encoding="utf-8-sig")

result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "task": PAYLOAD["task"],
    "source": PAYLOAD["source"],
    "training": {"seconds": training_seconds, "final_logged_loss": last_loss_log.get("loss"),
                 "last_logged_step": last_loss_log.get("step"), "steps": trainer.state.global_step,
                 "mean_training_loss": train_output.training_loss},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
result["groups"] = group_rows
if PAYLOAD["task"] == "glossary":
    result["search"] = search_summary
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))

if USE_DRIVE:
    import shutil
    for path in WORK.iterdir():
        if path.is_file():
            shutil.copy(path, Path(CONFIG["drive_dir"]) / path.name)
    print("Driveに保存しました:", CONFIG["drive_dir"])
print("保存先:", WORK)

,label,group,n,format_rate,content_rate
0,before,alias,12,0.750000,0.083333
1,before,canonical,12,0.583333,0.000000
2,before,paraphrase,12,0.500000,0.000000
3,before,unregistered,6,0.666667,0.000000
4,after,alias,12,0.916667,0.000000
5,after,canonical,12,1.000000,0.000000
6,after,paraphrase,12,0.916667,0.250000
7,after,unregistered,6,1.000000,0.000000
8,search,alias,12,1.000000,1.000000
9,search,canonical,12,1.000000,1.000000


{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 3,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.14.1+cu130"
  },
  "task": "glossary",
  "source": {
    "path": "data/glossary.json",
    "version": "2026-draft-1",
    "sha256": "2352911cf093636387cd7828afa59c7651c5adab05eaca860aae027508ac1452",
    "note": "全58語を学習。評価対象はKEYWORDSにある12語。用語の未学習汎化を測る実験ではない。"
  },
  "training": {
    "seconds": 244.6,
    "final_logged_loss": 0.569762897491455,
    "last_logged_step": 40,
    "steps": 48,
    "mean_training_loss": 1.0229051609834034
  },
  "method": "LoRA（量子化なし）",
  "before": {
    "label": "before",
    "n": 42,
    "format_rate": 0.619047619047619